In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

DATA_DIR = "/content/drive/My Drive/Colab Notebooks/Order_Forecasting/Processed"

meta = pd.read_parquet(f"{DATA_DIR}/val_metadata.parquet")

lgb_pred = pd.read_csv(f"{DATA_DIR}/pred_lightgbm.csv")["prediction"]
xgb_pred = pd.read_csv(f"{DATA_DIR}/pred_xgboost.csv")["prediction"]

y_true = meta["sales"]

In [ ]:
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSLE": rmsle(y_true, y_pred)
    }

results = [
    evaluate("LightGBM", y_true, lgb_pred),
    evaluate("XGBoost", y_true, xgb_pred),
]

results_df = pd.DataFrame(results).sort_values("RMSLE")
results_df

,model,RMSE,MAE,RMSLE
0,LightGBM,226.669618,59.597613,3.131095
1,XGBoost,227.534680,58.068428,3.134542


In [ ]:
best_model = results_df.iloc[0]["model"]
print(f"Best model based on RMSLE: {best_model}")

Best model based on RMSLE: LightGBM


In [ ]:
final_df = meta.copy()

final_df["pred_lightgbm"] = lgb_pred.values
final_df["pred_xgboost"]  = xgb_pred.values

In [ ]:
final_df["error_lgb"] = final_df["sales"] - final_df["pred_lightgbm"]
final_df["error_xgb"] = final_df["sales"] - final_df["pred_xgboost"]

In [ ]:
output_path = f"{DATA_DIR}/validation_predictions_all_models.csv"

final_df.to_csv(output_path, index=False)

print("File saved at:", output_path)

File saved at: /content/drive/My Drive/Colab Notebooks/Order_Forecasting/Processed/validation_predictions_all_models.csv
